In [19]:
import os
import pandas as pd
import sqlite3
from tqdm import tqdm

root_folder = 'C:/Users/20232075/Desktop/London Police Data'
db_path = 'crime_data.db'
batch_size = 2000

required_columns = [
    'Crime ID', 'Month', 'Reported by', 'Falls within',
    'Longitude', 'Latitude', 'Location',
    'LSOA code', 'Crime type', 'Last outcome category', 'Context'
]

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS crime (
        crimeID TEXT PRIMARY KEY,
        Month TEXT,
        Longitude REAL,
        Latitude REAL,
        LSOA_code TEXT,
        Type TEXT,
        Outcome TEXT
    )
''')
conn.commit()

insert_query = '''
    INSERT OR IGNORE INTO crime (
        crimeID, Month,Longitude, Latitude,
        LSOA_code, Type, Outcome
    ) VALUES (?, ?, ?, ?, ?, ?, ?);
'''

all_files = []
for subdir, dirs, files in os.walk(root_folder):
    for file in files:
        if file.endswith('.csv'):
            all_files.append(os.path.join(subdir, file))

batch = []
file_count = 0
inserted_rows = 0

for file_path in tqdm(all_files, desc="Processing files", unit="file"):
    name_without_ext = file_path[:-4]
    if name_without_ext.lower().endswith('-street'):
        df = pd.read_csv(file_path)
        for col in required_columns:
            if col not in df.columns:
                df[col] = None

        df = df.rename(columns={
            'Crime ID': 'crimeID',
            'Month': 'Month',
            'Longitude': 'Longitude',
            'Latitude': 'Latitude',
            'LSOA code': 'LSOA_code',
            'Crime type': 'Type',
            'Last outcome category': 'Outcome',
        })

        df = df[['crimeID', 'Month', 'Longitude', 'Latitude',
                  'LSOA_code', 'Type', 'Outcome',]]

        records = list(df.itertuples(index=False, name=None))

        for record in tqdm(records, desc=f"Inserting {os.path.basename(file_path)}", leave=False):
            batch.append(record)
            if len(batch) >= batch_size:
                cursor.executemany(insert_query, batch)
                conn.commit()
                inserted_rows += len(batch)
                batch = []

        file_count += 1

if batch:
    cursor.executemany(insert_query, batch)
    conn.commit()
    inserted_rows += len(batch)

conn.close()
print(f"\nInserted {inserted_rows} rows from {file_count} files.")


Processing files: 100%|██████████| 72/72 [03:18<00:00,  2.76s/file]


Inserted 3386817 rows from 36 files.


In [20]:

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

for file_path in tqdm(all_files, desc="Processing outcome files", unit="file"):
    name_without_ext = file_path[:-4]
    if name_without_ext.lower().endswith('-outcomes'):
        df = pd.read_csv(file_path)
        df = df[['Crime ID', 'Outcome type']].dropna(subset=['Crime ID'])

        update_records = list(df.itertuples(index=False, name=None))

        for crime_id, outcome in tqdm(update_records, desc=f"Updating {os.path.basename(file_path)}", leave=False):
            cursor.execute(
                "UPDATE crime SET Outcome = ? WHERE crimeID = ?;",
                (outcome, crime_id)
            )

conn.commit()
conn.close()
print("Outcome fields updated where applicable.")


Processing outcome files: 100%|██████████| 72/72 [01:12<00:00,  1.00s/file]


Outcome fields updated where applicable.


In [21]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) from crime"           )
result = cursor.fetchall()
print(result)

[(3309375,)]


In [22]:
import pandas as pd
import sqlite3
from tqdm import tqdm

tqdm.pandas()
db_path = 'crime_data.db'
ward_lsoa = "C:/Users/20232075/Downloads/LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.csv"

mapping_df = pd.read_csv(ward_lsoa, usecols=["LSOA21CD", "WD24CD"])

conn = sqlite3.connect(db_path)

df = pd.read_sql_query("SELECT * FROM crime", conn)

merged_df = df.merge(mapping_df, how='left', left_on='LSOA_code', right_on='LSOA21CD')

merged_df.drop(columns=['LSOA21CD'], inplace=True)

merged_df.to_sql("crime", conn, if_exists="replace", index=False)

conn.close()


In [24]:
import sqlite3

conn = sqlite3.connect("crime_data.db")
cursor = conn.cursor()

# Step 1: Get all existing non-integer crimeIDs
cursor.execute("SELECT crimeID FROM crime WHERE crimeID IS NOT NULL")
existing_ids = {row[0] for row in cursor.fetchall()}

# Step 2: Get rows with NULL crimeID
cursor.execute("SELECT rowid FROM crime WHERE crimeID IS NULL")
null_rows = [row[0] for row in cursor.fetchall()]

# Step 3: Start assigning numeric IDs (e.g., 1, 2, 3...) that don't clash with existing ones
new_id = 1
for rowid in null_rows:
    # Skip if that number is already used as a string
    while str(new_id) in existing_ids:
        new_id += 1
    cursor.execute("UPDATE crime SET crimeID = ? WHERE rowid = ?", (str(new_id), rowid))
    new_id += 1

conn.commit()
conn.close()


In [17]:
cursor.close()
conn.close()

ProgrammingError: Cannot operate on a closed database.

In [18]:
os.remove("crime_data.db")

In [26]:
conn = sqlite3.connect("crime_data.db")
cursor = conn.cursor()

cursor.execute("SELECT DISTINCT Type FROM crime")
crime_types = [row[0] for row in cursor.fetchall()]

print("Crime Types:")
for ctype in crime_types:
    print(ctype)

conn.close()


Crime Types:
Violence and sexual offences
Theft from the person
Anti-social behaviour
Burglary
Criminal damage and arson
Public order
Robbery
Vehicle crime
Drugs
Other theft
Shoplifting
Bicycle theft
Possession of weapons
Other crime


In [30]:
conn = sqlite3.connect("crime_data.db")

# Read directly into a pandas DataFrame
df = pd.read_sql_query("SELECT * FROM crime LIMIT 5", conn)

print(df)

conn.close()


                                             crimeID    Month  Longitude  \
0  6b1be5a8275fcda2982f6de7bb92e03499d33c7fa6316b...  2022-03  -0.445898   
1  b466ed98f60835f379377cda209f72a65f4b594d64b8a6...  2022-03  -0.530681   
2  cc667a5d8c2fa4f30ade960d334d4c1feac01170092470...  2022-03   0.876572   
3  a60d9d1cff047a67e9a22361a6aa27ad978ec0f71f482a...  2022-03   0.969845   
4  62823a01dfebac3445a665d4e0ebc92af01106c8384c26...  2022-03  -0.813313   

    Latitude  LSOA_code                          Type  \
0  50.803304  E01031422  Violence and sexual offences   
1  50.804178  E01031400  Violence and sexual offences   
2  51.137084  E01024001         Theft from the person   
3  51.124173  E01024013  Violence and sexual offences   
4  51.809402  E01017712  Violence and sexual offences   

                                         Outcome     WD24CD  
0                      Status update unavailable  E05009812  
1  Investigation complete; no suspect identified  E05009805  
2  Investigati